In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from flaml import AutoML

In [17]:
# 데이터 로드
df = pd.read_csv("../data/processed/hr_attrition_v2.csv")

# 타깃 / 입력변수 분리
target_col = "attrition_flag"
X = df.drop(columns=[target_col]).copy()
y = df[target_col].astype(int).copy()

print("원본 X shape:", X.shape)


원본 X shape: (1470, 35)


In [18]:
# 범주형 변수 확인
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
print("범주형 변수:", cat_cols)

# 원-핫 인코딩
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=False, dtype=int)

# 값 dtype 정리
X_encoded = X_encoded.astype(float)

# 컬럼명 백업 후 정수형으로 변경
original_feature_names = X_encoded.columns.tolist()
X_encoded.columns = range(X_encoded.shape[1])

print("인코딩 후 X shape:", X_encoded.shape)
print(X_encoded.dtypes.value_counts())
print("컬럼 인덱스 dtype:", X_encoded.columns.dtype)

# train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

범주형 변수: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'age_group', 'income_group', 'tenure_group']
인코딩 후 X shape: (1470, 67)
float64    67
Name: count, dtype: int64
컬럼 인덱스 dtype: int64
(1176, 67) (294, 67)
0.16156462585034015 0.1598639455782313


In [21]:
import warnings
warnings.filterwarnings("ignore")

In [24]:
# AutoML 설정
automl = AutoML()

settings = {
    "time_budget": 180,
    "metric": "roc_auc",
    "task": "classification",
    "estimator_list": ["lgbm", "rf", "extra_tree", "lrl1", "lrl2"],
    "eval_method": "cv",
    "n_splits": 5,
    "seed": 42,
    "log_file_name": "../outputs/flaml_automl.log",
    "verbose": 0,
}

# 학습
automl.fit(X_train=X_train, y_train=y_train, **settings)

In [25]:
# 결과 확인
print("Best estimator:", automl.best_estimator)
print("Best config:", automl.best_config)
print("Best CV loss:", automl.best_loss)

# 테스트 평가
y_pred = automl.predict(X_test)
y_prob = automl.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC:", average_precision_score(y_test, y_prob))
print()
print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification Report")
print(classification_report(y_test, y_pred, digits=4))

Best estimator: lgbm
Best config: {'n_estimators': 1175, 'num_leaves': 12, 'min_child_samples': 48, 'learning_rate': np.float64(0.11067058042633703), 'log_max_bin': 6, 'colsample_bytree': np.float64(0.6664720978221137), 'reg_alpha': np.float64(0.026494747355871394), 'reg_lambda': np.float64(0.036947966206460126)}
Best CV loss: 0.1684287437560213
ROC-AUC: 0.7737100525454389
PR-AUC: 0.4716695896397436

Confusion Matrix
[[237  10]
 [ 33  14]]

Classification Report
              precision    recall  f1-score   support

           0     0.8778    0.9595    0.9168       247
           1     0.5833    0.2979    0.3944        47

    accuracy                         0.8537       294
   macro avg     0.7306    0.6287    0.6556       294
weighted avg     0.8307    0.8537    0.8333       294



AutoML 결과 메모

- v2 변수셋 기준 AutoML 적용 결과, 최적 추정기는 lgbm으로 선택
- 테스트 성능은 로지스틱 회귀 baseline 대비 개선되지 않음
- 특히 이직자 recall이 낮아 HR 조기경보 목적에는 부적합
- 현재 데이터 구조에서는 복잡한 비선형 모델보다 해석 가능성과 안정성이 높은 로지스틱 회귀가 더 적합한 기준 모델로 판단
- 시사점 : 모델 복잡도보다 변수 설계와 문제 정의의 중요성 확인